# Laboratoire de recommandation JaimeTV

**Phase 1 :** explorer les métadonnées TMDB locales et établir une référence TF-IDF. La base SQLite est lue en mode seule lecture; aucun service externe ni export n’est utilisé. Le notebook s’exécute sur l’ordinateur local, pas sur la VM Oracle.

Le corpus utilise le titre, le résumé espagnol et les genres TMDB. TF-IDF classe les titres par mots communs; il ne comprend pas encore les filtres exacts ni les préférences subjectives. Les dix prompts restent en espagnol, langue prévue pour l’application.

## Plan et validations

1. **Environnement :** Python 3.12+; installer localement `requirements.txt` et `requirements-lab.txt`.
2. **Critères :** métadonnées actuelles, matrice TF-IDF valide, résultats uniques (maximum cinq) et scores finis. La pertinence humaine reste à évaluer.
3. **Structure :** réutiliser `catalog.fingerprint`; lire la base sans la modifier.
4. **Phase 1, MVP :** métadonnées TMDB → TF-IDF → classement de titres.
5. **Tests :** assertions et tests de fumée sur les dix prompts.
6. **Phase 2 :** modèle sémantique espagnol et filtres exacts; attendre la validation de la phase 1.
7. **Mesures :** comparer au baseline; mesurer qualité, latence et mémoire.
8. **Robustesse :** traiter les entrées non prises en charge et les erreurs sans interrompre le catalogue.
9. **Avancement :** le checklist distingue validation technique et revue humaine. Les phases 2 et 3 restent bloquées jusqu’à validation.

In [15]:
import json
import os
import random
import sqlite3
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((path for path in candidates if (path / "catalog.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Ouvrez ce notebook dans le workspace JaimeTV pour trouver catalog.py.")

configured_path = os.environ.get("IPTV_DB_PATH")
DATABASE_PATH = Path(configured_path).expanduser() if configured_path else PROJECT_ROOT / "catalog.db"
DATABASE_PATH = DATABASE_PATH.resolve()
if not DATABASE_PATH.is_file():
    raise FileNotFoundError(
        f"Base locale introuvable : {DATABASE_PATH}. Définissez IPTV_DB_PATH ou placez catalog.db dans le projet."
    )

sys.path.insert(0, str(PROJECT_ROOT))
from catalog import fingerprint

try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
except ModuleNotFoundError as error:
    raise ModuleNotFoundError(
        "scikit-learn manque dans cet environnement. Installez-le avec : "
        "python -m pip install -r requirements.txt -r requirements-lab.txt"
    ) from error

print(f"Projet : {PROJECT_ROOT}")
print("Base SQLite disponible; ouverture en lecture seule.")
print(f"Graine reproductible : {SEED}")

Projet : /Users/li23qf/Jaime/iptv
Base SQLite disponible; ouverture en lecture seule.
Graine reproductible : 42


In [22]:
CATALOGS = {
    "movies": ("movies", "stream_id"),
    "series": ("series", "series_id"),
}
COUNTRY = "CA"
matched_records = []
stale_matched = {kind: 0 for kind in CATALOGS}
raw_totals = {}
status_counts = {}

connection = sqlite3.connect(f"{DATABASE_PATH.as_uri()}?mode=ro", uri=True)
connection.row_factory = sqlite3.Row
try:
    tables = {row[0] for row in connection.execute("SELECT name FROM sqlite_master WHERE type='table'")}
    required_tables = {"tmdb_metadata", *(table for table, _ in CATALOGS.values())}
    if not required_tables.issubset(tables):
        missing = sorted(required_tables - tables)
        raise RuntimeError(f"Tables requises absentes : {missing}")

    for kind, (table, key) in CATALOGS.items():
        raw_totals[kind] = connection.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
        rows = connection.execute(
            f"""SELECT CAST(s.{key} AS TEXT) AS local_record_id, s.raw_json,
                       m.fingerprint, m.payload
                FROM {table} AS s
                JOIN tmdb_metadata AS m
                  ON m.kind = ? AND m.record_id = CAST(s.{key} AS TEXT) AND m.country = ?
                WHERE json_extract(m.payload, '$.status') = 'matched'""",
            (kind, COUNTRY),
        ).fetchall()

        for row in rows:
            raw = json.loads(row["raw_json"])
            if fingerprint(raw) != row["fingerprint"]:
                stale_matched[kind] += 1
                continue
            payload = json.loads(row["payload"])
            genres = payload.get("genres") if isinstance(payload.get("genres"), list) else []
            providers = payload.get("providers") if isinstance(payload.get("providers"), dict) else {}
            provider_names = sorted({
                str(name).strip()
                for names in providers.values() if isinstance(names, list)
                for name in names if isinstance(name, str) and name.strip()
            }, key=str.casefold)
            title = payload.get("title") if isinstance(payload.get("title"), str) else ""
            overview = payload.get("overview") if isinstance(payload.get("overview"), str) else ""
            genres = [name.strip() for name in genres if isinstance(name, str) and name.strip()]
            searchable_text = " ".join(part for part in (title, " ".join(genres), overview) if part).strip()
            matched_records.append({
                "kind": kind,
                "local_record_id": row["local_record_id"],
                "tmdb_id": payload.get("id"),
                "title": title,
                "overview": overview,
                "genres": genres,
                "providers": provider_names,
                "text": searchable_text,
            })

    for row in connection.execute("SELECT kind, payload FROM tmdb_metadata WHERE country=?", (COUNTRY,)):
        payload = json.loads(row["payload"])
        status = payload.get("status", "unknown")
        key = (row["kind"], status)
        status_counts[key] = status_counts.get(key, 0) + 1
finally:
    connection.close()

record_df = pd.DataFrame(matched_records)
if record_df.empty:
    catalog_df = pd.DataFrame(columns=[
        "kind", "tmdb_id", "local_record_ids", "title", "overview", "genres", "providers", "text"
    ])
else:
    usable_df = record_df[(record_df["tmdb_id"].notna()) & (record_df["title"].str.strip() != "")]
    catalog_rows = []
    for (kind, tmdb_id), group in usable_df.groupby(["kind", "tmdb_id"], sort=True):
        first = group.iloc[0]
        catalog_rows.append({
            "kind": kind,
            "tmdb_id": int(tmdb_id),
            "local_record_ids": sorted(group["local_record_id"].unique()),
            "title": first["title"],
            "overview": next((value for value in group["overview"] if value), ""),
            "genres": first["genres"],
            "providers": sorted({name for names in group["providers"] for name in names}, key=str.casefold),
            "text": next((value for value in group["text"] if value), first["title"]),
        })
    catalog_df = pd.DataFrame(catalog_rows)

kind_labels = {"movies": "Films", "series": "Séries"}
coverage_rows = []
for kind in CATALOGS:
    current_matches = sum(row["kind"] == kind for row in matched_records)
    unique_titles = int((catalog_df["kind"] == kind).sum()) if not catalog_df.empty else 0
    coverage_rows.append({
        "Type": kind_labels[kind],
        "Entrées locales": raw_totals.get(kind, 0),
        "Métadonnées TMDB valides": current_matches,
        "Caches périmées exclues": stale_matched[kind],
        "Titres TMDB uniques": unique_titles,
        "Copies regroupées": current_matches - unique_titles,
    })
coverage_df = pd.DataFrame(coverage_rows)
display(coverage_df)
status_labels = {
    "matched": "associé", "candidates": "candidats", "missing": "sans correspondance",
    "pending": "en attente", "unknown": "inconnu",
}
status_df = pd.DataFrame([
    {"Type": kind_labels.get(kind, kind), "Statut TMDB": status_labels.get(status, status), "Nombre": count}
    for (kind, status), count in sorted(status_counts.items())
])
display(status_df)
print(f"Titres TMDB uniques exploitables : {len(catalog_df)}")

,Type,Entrées locales,Métadonnées TMDB valides,Caches périmées exclues,Titres TMDB uniques,Copies regroupées
0,Films,6892,1092,0,1079,13
1,Séries,2358,4,0,4,0


,Type,Statut TMDB,Nombre
0,Films,candidats,77
1,Films,associé,1092
2,Films,sans correspondance,3
3,Séries,associé,4


Titres TMDB uniques exploitables : 1083


In [23]:
REFERENCE_TITLE = "Scary Movie"
reference_status = {
    "Entrées locales": 0,
    "Association TMDB valide": 0,
    "Autres statuts actuels": 0,
    "Cache périmé": 0,
    "Sans cache TMDB": 0,
}
connection = sqlite3.connect(f"{DATABASE_PATH.as_uri()}?mode=ro", uri=True)
connection.row_factory = sqlite3.Row
try:
    for kind, (table, key) in CATALOGS.items():
        rows = connection.execute(
            f"""SELECT s.raw_json, m.fingerprint, m.payload
                FROM {table} AS s
                LEFT JOIN tmdb_metadata AS m
                  ON m.kind=? AND m.record_id=CAST(s.{key} AS TEXT) AND m.country=?
                WHERE LOWER(s.name) LIKE ?""",
            (kind, COUNTRY, f"%{REFERENCE_TITLE.casefold()}%"),
        ).fetchall()
        for row in rows:
            reference_status["Entrées locales"] += 1
            if row["payload"] is None:
                reference_status["Sans cache TMDB"] += 1
                continue
            if fingerprint(json.loads(row["raw_json"])) != row["fingerprint"]:
                reference_status["Cache périmé"] += 1
                continue
            payload = json.loads(row["payload"])
            if payload.get("status") == "matched":
                reference_status["Association TMDB valide"] += 1
            else:
                reference_status["Autres statuts actuels"] += 1
finally:
    connection.close()

display(pd.DataFrame([reference_status], index=[REFERENCE_TITLE]))

,Entrées locales,Association TMDB valide,Autres statuts actuels,Cache périmé,Sans cache TMDB
Scary Movie,6,0,0,0,6


In [24]:
vectorizer = None
tfidf_matrix = None
baseline_ready = False

if not catalog_df.empty:
    vectorizer = TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1, 2),
        min_df=1,
        sublinear_tf=True,
        stop_words=None,  # Garde « no »; TF-IDF ne comprend toutefois pas la négation.
    )
    tfidf_matrix = vectorizer.fit_transform(catalog_df["text"].fillna(""))
    baseline_ready = tfidf_matrix.shape[0] == len(catalog_df) and tfidf_matrix.shape[1] > 0
    print(f"Matrice TF-IDF : {tfidf_matrix.shape[0]} titres × {tfidf_matrix.shape[1]} termes")
else:
    print("Aucun titre TMDB exploitable; matrice TF-IDF non créée.")

RESULT_COLUMNS = ["kind", "title", "genres", "score", "copies"]

def recommend_prompt(prompt, kind=None, top_n=5):
    """Baseline lexical : compare les mots du prompt aux métadonnées TMDB, sans filtres exacts."""
    if not baseline_ready:
        return pd.DataFrame(columns=RESULT_COLUMNS)
    query_vector = vectorizer.transform([str(prompt)])
    scores = cosine_similarity(query_vector, tfidf_matrix).ravel()
    candidates = np.arange(len(catalog_df))
    if kind is not None:
        candidates = candidates[catalog_df.iloc[candidates]["kind"].to_numpy() == kind]
    if not len(candidates):
        return pd.DataFrame(columns=RESULT_COLUMNS)
    ordered = candidates[np.argsort(-scores[candidates], kind="stable")[:max(0, top_n)]]
    rows = catalog_df.iloc[ordered]
    return pd.DataFrame({
        "kind": rows["kind"].to_numpy(),
        "title": rows["title"].to_numpy(),
        "genres": [", ".join(value) for value in rows["genres"]],
        "score": scores[ordered],
        "copies": [len(value) for value in rows["local_record_ids"]],
    }).reset_index(drop=True)


def recommend_similar_to_title(reference_title, kind=None, top_n=5):
    """Compare les métadonnées d’un titre de référence aux autres; correspondance exacte pour cette phase."""
    if not baseline_ready:
        return pd.DataFrame(columns=RESULT_COLUMNS)
    matches = catalog_df["title"].str.casefold() == str(reference_title).strip().casefold()
    if kind is not None:
        matches &= catalog_df["kind"] == kind
    reference_indices = np.flatnonzero(matches.to_numpy())
    if not len(reference_indices):
        return pd.DataFrame(columns=RESULT_COLUMNS)
    reference_index = reference_indices[0]
    scores = cosine_similarity(tfidf_matrix[reference_index], tfidf_matrix).ravel()
    candidates = np.arange(len(catalog_df))
    candidates = candidates[candidates != reference_index]
    if kind is not None:
        candidates = candidates[catalog_df.iloc[candidates]["kind"].to_numpy() == kind]
    ordered = candidates[np.argsort(-scores[candidates], kind="stable")[:max(0, top_n)]]
    rows = catalog_df.iloc[ordered]
    return pd.DataFrame({
        "kind": rows["kind"].to_numpy(),
        "title": rows["title"].to_numpy(),
        "genres": [", ".join(value) for value in rows["genres"]],
        "score": scores[ordered],
        "copies": [len(value) for value in rows["local_record_ids"]],
    }).reset_index(drop=True)

print("Baseline prêt :", baseline_ready)
print("Les scores classent les résultats; ils ne mesurent pas une pertinence calibrée.")

Matrice TF-IDF : 1083 titres × 48692 termes
Baseline prêt : True
Les scores classent les résultats; ils ne mesurent pas une pertinence calibrée.


### Dix prompts d’évaluation

Ces exemples servent à évaluer, pas à entraîner TF-IDF. Le baseline ne comprend pas « seulement », les plateformes, la négation ni les critères non pris en charge. Les prompts restent en espagnol, langue de l’application.

1. « Quiero solo series de ciencia ficción. »
2. « Recomiéndame películas de comedia disponibles en Netflix. »
3. « Busco algo de ciencia ficción disponible en Netflix. »
4. « Recomiéndame algo parecido a Scary Movie. »
5. « Quiero algo parecido a Scary Movie, pero no muy violento. »
6. « Quiero una película de misterio con un ambiente oscuro. »
7. « Dame una comedia de menos de 90 minutos. »
8. « Solo series estrenadas después de 2020. »
9. « Busco series de ciencia ficción disponibles en CineNube. »
10. « Recomiéndame algo parecido a La nebulosa de los lunes 9843. »

Réviser jusqu’à cinq titres par prompt. *Scary Movie* est le cas ambigu : la comédie et les mentions de violence ne permettent pas de conclure sur son niveau réel.

In [25]:
TEST_PROMPTS = [
    "Quiero solo series de ciencia ficción.",
    "Recomiéndame películas de comedia disponibles en Netflix.",
    "Busco algo de ciencia ficción disponible en Netflix.",
    "Recomiéndame algo parecido a Scary Movie.",
    "Quiero algo parecido a Scary Movie, pero no muy violento.",
    "Quiero una película de misterio con un ambiente oscuro.",
    "Dame una comedia de menos de 90 minutos.",
    "Solo series estrenadas después de 2020.",
    "Busco series de ciencia ficción disponibles en CineNube.",
    "Recomiéndame algo parecido a La nebulosa de los lunes 9843.",
]

prediction_rows = []
for prompt_id, prompt in enumerate(TEST_PROMPTS, start=1):
    results = recommend_prompt(prompt, top_n=5)
    for rank, result in results.iterrows():
        prediction_rows.append({
            "prompt_id": prompt_id,
            "prompt": prompt,
            "rank": rank + 1,
            "tipo": result["kind"],
            "titulo": result["title"],
            "generos": result["genres"],
            "score_tfidf": round(float(result["score"]), 4),
            "revision_humana": "",
            "notas": "",
        })

review_df = pd.DataFrame(prediction_rows, columns=[
    "prompt_id", "prompt", "rank", "tipo", "titulo", "generos",
    "score_tfidf", "revision_humana", "notas",
])
print(f"Candidats à examiner : {len(review_df)} (maximum 50)")
print("Étiquettes : pertinente / incertaine / non pertinente.")
review_columns = {
    "prompt_id": "ID", "prompt": "Prompt (espagnol)", "rank": "Rang",
    "tipo": "Type", "titulo": "Titre TMDB", "generos": "Genres TMDB (espagnol)",
    "score_tfidf": "Score TF-IDF", "revision_humana": "Évaluation humaine", "notas": "Notes",
}
review_display = review_df.rename(columns=review_columns).copy()
review_display["Type"] = review_display["Type"].map({"movies": "Films", "series": "Séries"})
display(review_display)

scary_exists = (
    not catalog_df.empty
    and catalog_df["title"].str.casefold().eq("scary movie").any()
)
if scary_exists:
    print("Titres similaires à Scary Movie selon ses métadonnées TMDB :")
    scary_results = recommend_similar_to_title("Scary Movie", top_n=5).rename(columns={
        "kind": "Type", "title": "Titre TMDB", "genres": "Genres TMDB (espagnol)",
        "score": "Score TF-IDF", "copies": "Copies locales",
    })
    scary_results["Type"] = scary_results["Type"].map({"movies": "Films", "series": "Séries"})
    display(scary_results)
else:
    print(
        "Scary Movie existe dans le catalogue, mais sans cache TMDB actuel. "
        "Consultez ce titre dans l’administration, puis relancez les cellules d’inventaire et de recommandation."
    )

Candidats à examiner : 50 (maximum 50)
Étiquettes : pertinente / incertaine / non pertinente.


,ID,Prompt (espagnol),Rang,Type,Titre TMDB,Genres TMDB (espagnol),Score TF-IDF,Évaluation humaine,Notes
0,1,Quiero solo series de ciencia ficción.,1,Films,2.0,"Acción, Ciencia ficción, Fantasía",0.2035,,
1,1,Quiero solo series de ciencia ficción.,2,Films,2:Hrs,"Familia, Aventura, Ciencia ficción",0.1581,,
2,1,Quiero solo series de ciencia ficción.,3,Films,Crónicas,"Animación, Ciencia ficción, Terror",0.1472,,
3,1,Quiero solo series de ciencia ficción.,4,Films,Absolutamente todo,"Comedia, Ciencia ficción",0.1323,,
4,1,Quiero solo series de ciencia ficción.,5,Films,Burn Out,"Animación, Familia, Ciencia ficción",0.1274,,
5,2,Recomiéndame películas de comedia disponibles ...,1,Films,Cinturones negros,Drama,0.1164,,
6,2,Recomiéndame películas de comedia disponibles ...,2,Films,Adam Sandler: 100% Fresh,"Comedia, Música",0.1156,,
7,2,Recomiéndame películas de comedia disponibles ...,3,Films,Bajo la sal,"Terror, Misterio, Suspense",0.1072,,
8,2,Recomiéndame películas de comedia disponibles ...,4,Films,Amor a quemarropa,"Acción, Crimen, Romance",0.0778,,
9,2,Recomiéndame películas de comedia disponibles ...,5,Films,Christmas Movie Magic,"Película de TV, Comedia, Romance",0.0775,,


Scary Movie existe dans le catalogue, mais sans cache TMDB actuel. Consultez ce titre dans l’administration, puis relancez les cellules d’inventaire et de recommandation.


### Étiquettes de revue

- **pertinente** : les métadonnées soutiennent la recommandation.
- **incertaine** : signaux contradictoires ou données insuffisantes.
- **non pertinente** : les métadonnées ne soutiennent pas la recommandation.

Les scores TF-IDF servent uniquement au classement; ce ne sont ni des probabilités ni des mesures calibrées. La qualité reste à valider manuellement.

In [26]:
# Après la revue, ajoute les étiquettes avec (prompt_id, type, titre) comme clé.
# Exemple : HUMAN_LABELS[(4, "movies", "Scary Movie")] = "incertaine"
HUMAN_LABELS = {}
ALLOWED_LABELS = {"pertinente", "incertaine", "non pertinente"}

if not review_df.empty:
    review_df["revision_humana"] = [
        HUMAN_LABELS.get((int(row.prompt_id), row.tipo, row.titulo), "")
        for row in review_df.itertuples(index=False)
    ]
invalid_labels = set(review_df["revision_humana"].dropna()) - {"", *ALLOWED_LABELS}
if invalid_labels:
    raise ValueError(f"Étiquettes inconnues : {sorted(invalid_labels)}")

reviewed_count = int(review_df["revision_humana"].isin(ALLOWED_LABELS).sum()) if not review_df.empty else 0
print(f"Résultats évalués : {reviewed_count} sur {len(review_df)}")
review_columns = {
    "prompt_id": "ID", "prompt": "Prompt (espagnol)", "rank": "Rang",
    "tipo": "Type", "titulo": "Titre", "generos": "Genres",
    "score_tfidf": "Score TF-IDF", "revision_humana": "Évaluation humaine", "notas": "Notes",
}
display(review_df[review_df["revision_humana"] != ""].rename(columns=review_columns))

Résultats évalués : 0 sur 50


,ID,Prompt (espagnol),Rang,Type,Titre,Genres,Score TF-IDF,Évaluation humaine,Notes


In [27]:
smoke_results = recommend_prompt("comedia de ciencia ficción", top_n=5)
smoke_scores_finite = (
    bool(np.isfinite(smoke_results["score"].to_numpy()).all()) if not smoke_results.empty else False
)
smoke_titles_unique = (
    not smoke_results["title"].duplicated().any() if not smoke_results.empty else False
)
smoke_top_five = len(smoke_results) <= 5

if not catalog_df.empty:
    reference_sample = catalog_df.iloc[0]
    similarity_results = recommend_similar_to_title(
        reference_sample["title"], kind=reference_sample["kind"], top_n=5
    )
    similarity_smoke_valid = (
        len(similarity_results) <= 5
        and reference_sample["title"] not in set(similarity_results["title"])
        and (similarity_results.empty or np.isfinite(similarity_results["score"].to_numpy()).all())
    )
else:
    similarity_smoke_valid = False
    similarity_results = pd.DataFrame(columns=RESULT_COLUMNS)

phase1_checks = {
    "Base locale détectée": DATABASE_PATH.is_file(),
    "Titres avec métadonnées TMDB actuelles": not catalog_df.empty,
    "Matrice TF-IDF valide": baseline_ready,
    "Top-5 du prompt généré": smoke_top_five and not smoke_results.empty,
    "Scores finis": smoke_scores_finite,
    "Titres sans doublons": smoke_titles_unique,
    "Similarité à un titre testée": similarity_smoke_valid,
}
phase1_technical_passed = all(phase1_checks.values())
phase1_human_review_complete = (
    not review_df.empty
    and review_df["revision_humana"].isin(ALLOWED_LABELS).all()
)

checklist = pd.DataFrame([
    {"Critère": name, "État": "RÉUSSI" if passed else "BLOQUÉ"}
    for name, passed in phase1_checks.items()
])
checklist.loc[len(checklist)] = {
    "Critère": "Évaluation humaine des résultats",
    "État": "RÉUSSIE" if phase1_human_review_complete else "EN ATTENTE",
}
checklist.loc[len(checklist)] = {
    "Critère": "Phase 1 technique",
    "État": "RÉUSSIE" if phase1_technical_passed else "BLOQUÉE",
}
checklist.loc[len(checklist)] = {
    "Critère": "Phase 2 (modèle sémantique et filtres exacts)",
    "État": "BLOQUÉE jusqu’à la revue de cette phase",
}
display(checklist)
print("Résultats du test de requête :")
smoke_display = smoke_results.rename(columns={
    "kind": "Type", "title": "Titre TMDB", "genres": "Genres TMDB (espagnol)",
    "score": "Score TF-IDF", "copies": "Copies locales",
}).copy()
if not smoke_display.empty:
    smoke_display["Type"] = smoke_display["Type"].map({"movies": "Films", "series": "Séries"})
display(smoke_display)
print(f"Référence du test de similarité : {reference_sample['title'] if not catalog_df.empty else 'aucune'}")
similarity_display = similarity_results.rename(columns={
    "kind": "Type", "title": "Titre TMDB", "genres": "Genres TMDB (espagnol)",
    "score": "Score TF-IDF", "copies": "Copies locales",
}).copy()
if not similarity_display.empty:
    similarity_display["Type"] = similarity_display["Type"].map({"movies": "Films", "series": "Séries"})
display(similarity_display)

,Critère,État
0,Base locale détectée,RÉUSSI
1,Titres avec métadonnées TMDB actuelles,RÉUSSI
2,Matrice TF-IDF valide,RÉUSSI
3,Top-5 du prompt généré,RÉUSSI
4,Scores finis,RÉUSSI
5,Titres sans doublons,RÉUSSI
6,Similarité à un titre testée,RÉUSSI
7,Évaluation humaine des résultats,EN ATTENTE
8,Phase 1 technique,RÉUSSIE
9,Phase 2 (modèle sémantique et filtres exacts),BLOQUÉE jusqu’à la revue de cette phase


Résultats du test de requête :


,Type,Titre TMDB,Genres TMDB (espagnol),Score TF-IDF,Copies locales
0,Films,2.0,"Acción, Ciencia ficción, Fantasía",0.249816,1
1,Films,2:Hrs,"Familia, Aventura, Ciencia ficción",0.194150,1
2,Films,Crónicas,"Animación, Ciencia ficción, Terror",0.180672,1
3,Films,Absolutamente todo,"Comedia, Ciencia ficción",0.167307,1
4,Films,Burn Out,"Animación, Familia, Ciencia ficción",0.156397,1


Référence du test de similarité : American Beauty


,Type,Titre TMDB,Genres TMDB (espagnol),Score TF-IDF,Copies locales
0,Films,Aprender a soltar,Drama,0.064336,1
1,Films,10 vidas,"Animación, Comedia, Familia, Fantasía",0.048882,1
2,Films,23 000 vidas,Drama,0.048782,1
3,Films,Colisión,"Suspense, Crimen, Drama",0.045002,1
4,Films,Delitos y faltas,"Comedia, Drama, Crimen",0.042785,1
